# ERB · Kaggle runner (free T4 x2)
1. Locally: `python scripts/make_kaggle_bundle.py` → `erb2_bundle.zip`.
2. Kaggle → *Datasets → New dataset* → upload the zip (name it `erb2-bundle`).
3. New notebook → *Add input* → that dataset; *Settings → Accelerator → GPU T4 x2*; Internet **on**.
4. Run all. Download `/kaggle/working/erb2_results.zip` and unzip into the repo's `results/`.

T4 has no bf16: models load in fp16 with SDPA attention. The run resumes from `results/*.jsonl`, so a
session that hits the 12 h limit can be continued by re-running with the previous results attached.

In [ ]:
import os, glob, shutil, subprocess, zipfile
# Kaggle mounts datasets at varying depths (e.g. /kaggle/input/datasets/<user>/<name>/), and may
# auto-extract the zip. Walk the whole input tree for either the zip or an extracted folder with src/.
WORK = "/kaggle/working/erb2"
zip_path, folder = None, None
for root, dirs, files in os.walk("/kaggle/input"):
    if "erb2_bundle.zip" in files:
        zip_path = os.path.join(root, "erb2_bundle.zip"); break
    if "src" in dirs and os.path.exists(os.path.join(root, "src", "04_run_models.py")):
        folder = root; break
if zip_path:
    zipfile.ZipFile(zip_path).extractall(WORK)
elif folder:
    shutil.copytree(folder, WORK, dirs_exist_ok=True)
else:
    raise SystemExit("bundle not found under /kaggle/input - check that the dataset is attached")
os.chdir(WORK); print("bundle from", zip_path or folder); print(os.listdir())

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate bitsandbytes num2words pyyaml pyarrow
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Only the model that does not fit the 6 GB laptop. The others run locally (Qwen-3B there in 4-bit).
MODELS = ["internvl3-2b"]   # optional stretch: + "qwen2.5-vl-7b" (4-bit)

In [ ]:
# Smoke test + timing (20 items per dataset, ~1-2 min). Check: n_errors 0, note the s/item line.
!python src/04_run_models.py --model internvl3-2b --limit 20 --blocks main --tag smoke

In [ ]:
for m in MODELS:
    print("=" * 30, m)
    subprocess.run(["python", "src/04_run_models.py", "--model", m], check=False)

In [ ]:
with zipfile.ZipFile("/kaggle/working/erb2_results.zip", "w") as z:
    for f in glob.glob("results/*"):
        if "smoke" not in f:
            z.write(f)
print(os.listdir("results"))